# Kilonova End-to-End Simulation

Simulates a GW170817-like kilonova population (`KilonovaCoolingBlackbodySED`): a constant volumetric rate of 53 Gpc⁻³ yr⁻¹ (Fishbach et al. 2026) out to z=0.2, with a 30-day duration window.

This notebook is fully self-contained: it downloads the default UVEX schedule, samples a Monte
Carlo `Kilonova` population against it, screens the population down to what UVEX would
actually detect, and plots the result -- the same pipeline the `uvex-transients` CLI runs from a
config file (see `configs/quickstart_tde.yaml`, `configs/full_run.yaml`), just driven from Python
so you can poke at every intermediate object.

Run all cells top to bottom.


## Setup

Load the default schedule and configure the transient population.


In [ ]:
import numpy as np
from astropy import units as u
from matplotlib import pyplot as plt

# `uvex_transients` installs warning filters for some noisy-but-harmless dependency
# warnings (e.g. lal's Jupyter/IPython SWIG redirect notice) as soon as it's imported
# -- import it before `m4opt`/`ligo.skymap` (which trigger that warning) so the filter
# is already in place.
from uvex_transients.simulation.core import SurveySimulator
from uvex_transients.surveys import get_schedule
from uvex_transients.transients.kilonovae import Kilonova

from m4opt.missions import uvex

schedule = get_schedule()
kne = Kilonova()
simulator = SurveySimulator(schedule, transients={"kne": kne}, simulation_seed=42)

print(kne)
print(f"redshift_limit={kne.redshift_limit}, duration_limit={kne.duration_limit}")

## Sample a Monte Carlo population

`generate_events` uses **windowed sampling**: it only draws events within HEALPix pixels/time
bins the schedule could plausibly have caught, given the transient's own duration window.


In [ ]:
# Kilonovae are rare and confined to z<0.2 (small comoving volume), so a freshly-sampled catalog is already small -- no downsampling needed.
TIME_BINS = 200
NSIDE = 256
DOWNSAMPLE = None

catalog = simulator.generate_events(time_bins=TIME_BINS, nside=NSIDE, downsample=None)
print(f"Sampled {len(catalog) * (DOWNSAMPLE or 1)} events across {TIME_BINS} time bin(s) at NSIDE={NSIDE}.")

## Screen the population

Two progressively more expensive passes narrow the freshly-sampled catalog down to what matters:
first, a cheap check of whether an event could *ever* clear a fixed magnitude limit
(`filter_by_limiting_magnitude`); then the real question of whether it was actually detected
above a given SNR by an observation the schedule made (`filter_by_snr`).


In [ ]:
MAG_LIMIT = 25.0
SNR_THRESHOLD = 5.0

mag_filtered = simulator.filter_by_limiting_magnitude(catalog, uvex, mag_limit=MAG_LIMIT)
print(f"{len(mag_filtered) * (DOWNSAMPLE or 1)} could ever clear {MAG_LIMIT} AB mag.")

detected = simulator.filter_by_snr(mag_filtered, uvex, snr_threshold=SNR_THRESHOLD)
print(f"{len(detected) * (DOWNSAMPLE or 1)} were detected above SNR={SNR_THRESHOLD}.")

## Detection funnel


In [ ]:
stages = ["Sampled", f"Mag < {MAG_LIMIT}", f"SNR > {SNR_THRESHOLD}"]
counts = [
    len(catalog) * (DOWNSAMPLE or 1),
    len(mag_filtered) * (DOWNSAMPLE or 1),
    len(detected) * (DOWNSAMPLE or 1),
]

fig, ax = plt.subplots()
ax.bar(stages, counts, color=["#888888", "#4C72B0", "#55A868"])
for i, count in enumerate(counts):
    ax.text(i, count, f"{count:,}", ha="center", va="bottom")
ax.set_ylabel("Number of events")
ax.set_title("Detection funnel")
fig.tight_layout()

## Yield summary

Tabulate the population's footprint-aware exposure (`SurveySimulator.compute_effective_exposure`,
an `ExposureCatalog`), then combine it with `catalog`/`detected` into a yield estimate
(`EventCatalog.compute_yield_summary`, a `YieldTable`) -- rate, intrinsic UVEX event count,
detection probability, and expected detections, each with both Clopper-Pearson (Monte Carlo) and
rate-normalization confidence bounds. This is the same summary the `uvex-transients run` CLI
command writes to `exposure.ecsv`/`yield_summary.ecsv`/`yield_summary.txt`.
`YieldTable.display_expected_detections` renders just the expected-detections estimate, with both
uncertainties stacked, as typeset LaTeX.


In [ ]:
exposure = simulator.compute_effective_exposure(time_bins=TIME_BINS, nside=NSIDE)
yield_table = catalog.compute_yield_summary(detected, exposure, {"kne": kne})
yield_table.display_expected_detections()

## Sky distribution


In [ ]:
fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(111, projection="aitoff")
ax.grid(True)

ra_sampled = catalog.coord.ra.wrap_at(180 * u.deg).radian
ra_detected = detected.coord.ra.wrap_at(180 * u.deg).radian

ax.scatter(
    ra_sampled,
    catalog.coord.dec.radian,
    s=2,
    alpha=0.2,
    color="#888888",
    label=f"Sampled ({(DOWNSAMPLE or 1) * len(catalog)})",
)
ax.scatter(
    ra_detected,
    detected.coord.dec.radian,
    s=4,
    color="#55A868",
    label=f"Detected ({(DOWNSAMPLE or 1) * len(detected)})",
)
ax.legend(loc="lower right", markerscale=4)
ax.set_title("Sky distribution")
fig.tight_layout()

## Redshift distribution


In [ ]:
bins = np.linspace(0, catalog.redshift.max(), 30)

fig, ax = plt.subplots()
ax.hist(catalog.redshift, bins=bins, color="#888888", label=f"Sampled ({len(catalog)})")
ax.hist(detected.redshift, bins=bins, color="#55A868", label=f"Detected ({len(detected)})")
ax.set_yscale("log")
ax.set_xlabel("Redshift")
ax.set_ylabel("Number of events")
ax.legend()
fig.tight_layout()

## An example light curve

Reconstruct one event as a real `Event` (`EventCatalog.get_events`) and run full synthetic
photometry (`Event.simulate_photometry`) against every observation the schedule actually made of
it. Prefer a detected event, but this population's detected count depends on the specific random
draw above -- fall back to the best available stage (mag-screened, then the raw sample) if
`detected` came up empty.


In [ ]:
rng = np.random.default_rng(1)
for _label, _source in (("detected", detected), ("mag-screened", mag_filtered), ("sampled", catalog)):
    if len(_source) > 0:
        example_catalog, example_stage = _source, _label
        break
else:
    raise RuntimeError("No events at all were sampled -- try a larger TIME_BINS/NSIDE or a smaller DOWNSAMPLE.")

example_id = rng.choice(example_catalog.event_id)
event = example_catalog.get_events(int(example_id), {"kne": kne}, schedule)
print(f"Example event (from the {example_stage!r} stage):")
print(event)

phot = event.simulate_photometry(uvex)
t_since_explosion = (phot["obs_time"] - event.t_explosion).to(u.day)
t_theory = np.linspace(0, kne.duration_limit.to_value(u.day), 300) * u.day

fig, ax = plt.subplots(figsize=(7, 4))
for band, color in {"FUV": "#4C72B0", "NUV": "#DD8452"}.items():
    ax.plot(t_theory.value, event.mag(t_theory, uvex, band=band).value, color=color, lw=1.5, alpha=0.6)

    in_band = np.isfinite(phot["ab_mag"]) & (phot["band"] == band)
    detected_pts = in_band & (phot["snr"] > SNR_THRESHOLD)
    upper_limits = in_band & (phot["snr"] <= SNR_THRESHOLD)

    if np.any(detected_pts):
        ax.errorbar(
            t_since_explosion[detected_pts].value,
            phot["ab_mag"][detected_pts],
            yerr=5 * phot["mag_err"][detected_pts],
            marker="s",
            mfc=color,
            mec="k",
            ecolor=color,
            linestyle="none",
            label=band,
        )
    if np.any(upper_limits):
        ax.errorbar(
            t_since_explosion[upper_limits].value,
            phot["ab_mag"][upper_limits],
            yerr=[
                phot["mag_upper"][upper_limits] - phot["ab_mag"][upper_limits],
                np.abs(phot["mag_lower"][upper_limits] - phot["ab_mag"][upper_limits]),
            ],
            marker="v",
            mfc="w",
            mec=color,
            ecolor=color,
            linestyle="none",
        )

ax.invert_yaxis()
ax.set_xlabel("Days since explosion")
ax.set_ylabel("AB magnitude")
ax.set_title(f"Event {event.event_id} (z={event.redshift:.3f}, {event.n_observations} observations)")
ax.legend()
fig.tight_layout()
plt.show()